*   This notebook explains how values flow forward through a network and how  gradients flow backward through the recorded computation graph.

In [1]:
import math
import random
import numpy as np
import torch

# 5.3.1 Forward Propagation

## 1. Intuition

* Forward propagation is the process of computing outputs from inputs by running the model operations in order.

* Intermediate values are values created between input and output, such as hidden activations.

## 2. Why this exists

* The loss cannot be computed until predictions exist, and predictions come from the forward pass.

## 3. Examples

* A tiny forward pass through one hidden layer.

In [2]:
X = torch.tensor([[1.0, 2.0]])
W1 = torch.randn(2, 3)
b1 = torch.zeros(3)
W2 = torch.randn(3, 1)
b2 = torch.zeros(1)
H = torch.relu(X @ W1 + b1) # shapes of (1, 2) @ (2, 3) = shape of (1, 3)
y_hat = H @ W2 + b2 # shapes of (1, 3) @ (3, 1) = shape of (1, 1)
y_hat

tensor([[-3.6031]])

## 4. Step-by-step breakdown

* The input `X` is used first.

* `X @ W1 + b1` computes hidden pre-activations.

* `torch.relu` creates hidden activations and introduces nonlinearity into the model.

* `H @ W2 + b2` computes the final output.

* The forward pass stores values that backpropagation may need later.

## 5. Connection to ML systems

*   Training loops begin each step with forward propagation, then use the result to compute loss.

## 6. Common confusion points

- Forward propagation computes predictions only.
- It does not update parameters by itself.
- Intermediate values matter for gradient computation.
- Layer order determines computation order.

# 5.3.2 Computational Graph of Forward Propagation

## 1. Intuition

* A computational graph records how values are produced from other values.

* Nodes are values or operations. Edges show dependency: which value was used to make another value.

    ```
    X --> X @ W1 + b1 --> ReLU --> H --> H @ W2 + b2 --> y_hat
          ↑                  ↑             ↑
          W1,b1              operation     W2,b2
    ```

## 2. Why this exists

* The graph lets automatic differentiation apply the chain rule systematically.

* A computational graph isn't primarily a picture. It's a record of the computation's dependencies that autograd can traverse backward to calculate gradients.

## 3. Examples

* A small graph-like dependency list.

In [3]:
graph = [
    "X -> hidden linear",
    "hidden linear -> ReLU",
    "ReLU -> output linear",
    "output linear -> loss",
]

* PyTorch records tensor operations when gradients are required.


In [4]:
x = torch.tensor(2.0, requires_grad=True)
y = x * x
z = y + 3
z.grad_fn is not None # True because z was produced by an operation involving a tensor that requires gradients

True

## 4. Step-by-step breakdown

* The dependency list is a human-readable graph.

* In PyTorch, `requires_grad=True` asks autograd to track operations involving `x`.

* `y` and `z` remember how they were computed (inspectable via `grad_fn`).

* `grad_fn` is PyTorch's stored backward-function reference for non-leaf (not created directly by user) tensors.
> * A leaf tensor is a tensor created directly by the user.
> * A non-leaf tensor is a tensor created as the result of an operation/function.

## 5. Connection to ML systems

Deep learning frameworks build these graphs dynamically during ordinary tensor computation.

## 6. Common confusion points

- A graph is about dependencies, not visual appearance.
- Leaf tensors are created directly by the user.
- Non-leaf tensors are produced by operations.
- The graph is used for gradients after the loss is computed.

# 5.3.3 Backpropagation

## 1. Intuition

* Backpropagation computes gradients by moving backward from the loss through the computational graph.

* It applies the chain rule: multiply local sensitivities along dependency paths.

## 2. Why this exists

* Parameters can be improved only if we know how changing them would change the loss.

## 3. Examples

* PyTorch backpropagation through a small computation.

In [8]:
x = torch.tensor(2.0, requires_grad=True)
y = x * x
loss = y + 1 # loss = 5
loss.backward() # derivative of x² + 1 at x=2 is 4
x.grad

False


tensor(4.)

*   Manual chain rule for the same computation.

In [9]:
x_value = 2.0
dloss_by = 1.0
dy_dx = 2 * x_value # dy_dx = 2 * 2 = 4, where y is calculated
dloss_dx = dloss_by * dy_dx # dloss_dx = 1 * 4 = 4, the gradient that loss.backward() computes
dloss_dx

4.0

## 4. Step-by-step breakdown

* `loss.backward()` starts from the scalar loss.

* PyTorch traces back through `loss = y + 1` and `y = x * x`.

* `x.grad` stores the derivative of loss with respect to `x`.

* The manual calculation multiplies the local derivative from each step.

## 5. Connection to ML systems

* Backpropagation is the core algorithm that makes neural network training practical.

## 6. Common confusion points

- Backpropagation computes gradients, not parameter updates.
- The optimizer uses gradients after backpropagation.
- Gradients accumulate unless cleared.
- The chain rule is the mathematical basis.

# 5.3.4 Training Neural Networks

## 1. Intuition

* Training a neural network repeats **forward propagation**, **loss computation**, **backpropagation**, and **param updates** (`optimizer.step()`).

* A parameter update changes weights and biases using an optimizer.

## 2. Why this exists

* The full loop is where all pieces connect into learning.

## 3. Examples

* One tiny neural-network training step.

In [ ]:
# Initialization
net = torch.nn.Sequential(
    torch.nn.Linear(2, 3),                           # First linear layer: 2 inputs → 3 outputs
    torch.nn.ReLU(),                                 # Sets negative activations to 0
    torch.nn.Linear(3, 1))                           # Final linear layer: 3 inputs → 1 output
X = torch.randn(4, 2)
y = torch.randn(4, 1)
loss_fn = torch.nn.MSELoss()                         # Loss will be calculated via MSELoss
opt = torch.optim.SGD(                               # Optimizer is SGD
    net.parameters(),                                # params that SGD will upgrade come from net
    lr=0.01)

opt.zero_grad()                                      # Clear gradients from any previous training step
loss = loss_fn(net(X), y)                            # Forward pass: produce predictions
loss.backward()                                      # Backpropagation: compute gradients of loss w.r.t. parameters
opt.step()                                           # Apply gradient updates to params (W1, b1 in the first input layer, and W2, b2 in the final output layer)

loss

## 4. Step-by-step breakdown

* The model computes predictions with `net(X)`.

* The loss compares predictions with labels.

* `zero_grad()` clears old gradients.

* `backward()` computes new gradients.

* `step()` updates parameters.

## 5. Connection to ML systems

*   This control flow is shared by MLPs, CNNs, RNNs, and transformers, even though their layers differ.

      ```
      Neural network architectures
      │
      ├── MLP
      │   └── general-purpose vector transformations
      │
      ├── CNN
      │   └── local/shared patterns
      │
      ├── RNN
      │   └── sequential processing + hidden state
      │
      └── Transformer
          └── attention-based interactions
      ```



## 6. Common confusion points

- Training is a repeated process, not one forward pass.
- Clearing gradients is necessary before the next backward pass.
- Loss must connect to parameters through the graph.
- Optimizer steps should happen after gradients exist.

# 5.3.5 Summary

## 1. Intuition

* Forward propagation computes **predictions**. Backpropagation computes **gradients**. The optimizer updates **parameters**.

* The computational graph connects these steps.

## 2. Why this exists

* Understanding this flow removes much of the mystery from deep learning training loops.

## 3. Examples

* The repeated training sequence.

In [ ]:
sequence = [
    "forward",
    "loss",
    "zero gradients",
    "backward",
    "optimizer step",
]

## 4. Step-by-step breakdown

* The sequence names what happens in one training iteration.

* In many PyTorch loops, `zero_grad` appears before forward or before backward.

* The important rule is that old gradients should not pollute the new update.

* Old gradients can contaminate a new backward pass, but they do not contaminate the forward pass.

    ```
    Forward pass:

    X + W1 + W2
        ↓
      loss

    W1.grad ──────────┐
    W2.grad ──────────┤  ← ignored during forward
                      │
                      ↓
                  loss.backward()
    ```



## 5. Connection to ML systems

* Later training frameworks package this sequence, but the same execution order remains underneath.

## 6. Common confusion points

- The graph records how tensors were computed.
- Backward uses the graph to compute gradients.
- The optimizer changes parameters.
- Debug training by checking each step separately.

# 5.3.6 Exercises

## 1. Intuition

* These exercises practice tracing values and gradients.

## 2. Why this exists

* Small graphs make it easier to understand large neural networks.

## 3. Examples

* Exercise 1: compute a gradient through 2 operations.

In [10]:
x = torch.tensor(3.0, requires_grad=True)
y = 2 * x # 3 * 2 = 6
loss = y * y # 6 * 6 = 36
loss.backward() # dloss/dy = 2y, or 2 * 6 = 12 --> dy/dx = 2 --> dloss/dx = 12 * 2 = 24
x.grad

tensor(24.)

* Exercise 2: list the training loop steps from memory.

In [11]:
steps = [
    "forward pass",
    "calculate loss (e.g. MSELoss, CrossEntropy)",
    "loss.backward()",
    "optimizer.step()",]

## 4. Step-by-step breakdown

* Exercise 1 checks chain-rule tracking.

* Exercise 2 checks control-flow memory.

* Both are core skills for reading training code.

## 5. Connection to ML systems

* These same ideas appear in all later deep learning chapters.

## 6. Common confusion points

- Gradients are local to the current graph and values.
- A scalar loss makes `.backward()` straightforward because there is one objective to differentiate.
> * Most transformers are also trained with scalar loss.
- Parameter updates are separate from gradient computation.
- Always know what is stored in memory during training.